In [1]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict,Annotated
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class StructureEssay(BaseModel):
    feedback:str=Field(description="give me detail feedback of the essay ")
    score:int=Field(description="rate the essay out of 10",le=10,ge=0)


In [5]:
model_structure=model.with_structured_output(StructureEssay)

In [6]:
class upscState(TypedDict):
    essay:str
    deth_feedback:str
    clarity_feedback:str
    language_feedback:str
    individual_score:Annotated[list[int],operator.add]
    final_feedback:str
    final_score:float

In [7]:
graph=StateGraph(upscState)

In [8]:
def depth_evaluation(state:upscState):
    prompt=f"generate a feeback of the essay {state["essay"]} on the basis of the depth analysis and give me a score out of 10 "
    output=model_structure.invoke(prompt)
    return {"deth_feedback":output.feedback,"individual_score":[output.score]}

In [9]:
def clarity_evaluation(state:upscState):
    prompt=f"generate a feeback of the essay {state["essay"]} on the basis of the clarity and give me a score out of 10 "
    output=model_structure.invoke(prompt)
    return {"clarity_feedback":output.feedback,"individual_score":[output.score]}

In [10]:
def language_evaluation(state:upscState):
    prompt=f"generate a feeback of the essay {state["essay"]} on the basis of the language and give me a score out of 10 "
    output=model_structure.invoke(prompt)
    return {"language_feedback":output.feedback,"individual_score":[output.score]}

In [11]:
def final_evaluation(state:upscState):
    prompt=f"give me a final feedback of language_feedback{state["language_feedback"]}, clarity_feedback {state["clarity_feedback"]}, depth_feedback {state["deth_feedback"]}"
    final_feedback=model.invoke(prompt)
    prompt_sc=f"generate a final score (average score) from the list of score-> {state["individual_score"]}"
    final_score=model.invoke(prompt_sc)
    return {"final_score":final_score.content,"final_feedback":final_feedback.content}

In [12]:
graph.add_node("deth_evaluation",depth_evaluation)
graph.add_node("clarity_evaluation",clarity_evaluation)
graph.add_node("language_evaluation",language_evaluation)
graph.add_node("final_evaluation",final_evaluation)

graph.add_edge(START,"deth_evaluation")
graph.add_edge(START,"clarity_evaluation")
graph.add_edge(START,"language_evaluation")

graph.add_edge("deth_evaluation","final_evaluation")
graph.add_edge("clarity_evaluation","final_evaluation")
graph.add_edge("language_evaluation","final_evaluation")

graph.add_edge("final_evaluation",END)


In [13]:
workflow=graph.compile()

In [14]:
essay="""in an age obsessed with grand innovations—artificial intelligence, space exploration, and genetic engineering—we rarely pause to examine the quiet, invisible structures that shape our everyday existence. From the layout of a grocery store designed to maximize impulse buying to the algorithmically curated social media feed that subtly dictates our mood, modern life is governed by what architect Christopher Alexander called a “pattern language.” These unseen architectures, while often benign, warrant closer scrutiny because they influence our choices, constrain our autonomy, and ultimately redefine what it means to act freely.

First, consider the physical spaces we inhabit without thought. The suburban cul-de-sac, celebrated for its safety and community feel, simultaneously isolates residents by discouraging through-traffic and removing the chance encounters that define urban neighborhoods. Similarly, the modern open-plan office, intended to foster collaboration, often generates a tyranny of ambient noise and performative busyness. In each case, the design precedes and dictates behavior. People do not choose to be distracted; their environment decides for them. A 2019 study from the University of California, Irvine, found that after an interruption, office workers took an average of twenty-three minutes to return to their original task. That lost time is not a personal failing, but an architectural one.

Second, the digital realm amplifies this phenomenon with unprecedented precision. Recommendation engines on streaming platforms or e-commerce sites do not merely suggest—they steer. When Netflix’s algorithm highlights a thriller over a documentary, or when Amazon prioritizes a bestselling gadget over a niche tool, the outcome is a homogenization of taste. Worse, these systems are optimized for engagement, not enrichment. Clickbait headlines, infinite scrolling, and push notifications are not bugs; they are features designed to exploit psychological vulnerabilities. The philosopher C. Thi Nguyen has argued that such systems create “value capture,” where users forget their original intentions and instead chase the metrics the system rewards—likes, views, or purchases.

However, to recognize this subtle coercion is not to embrace a dystopian fatalism. Awareness itself is a countermeasure. By deliberately choosing discomfort—taking a different route to work, reading a physical newspaper instead of an aggregated feed, or disabling notifications for a single afternoon—individuals can reclaim small pockets of agency. More importantly, collective action can reshape the architecture. Public square designs that prioritize benches and fountains over retail kiosks, or social media platforms that default to chronological rather than algorithmic timelines, demonstrate that alternative patterns are possible.

In conclusion, the most powerful forces in our lives are often those we fail to notice. The layout of a room, the rhythm of a notification, the default setting on an app—these are the silent legislators of our daily choices. By learning to read the unseen architecture around us, we move from passive inhabitants to active designers. And in that shift lies the only true freedom our constructed world can offer: the freedom to see the cage, and then to decide, deliberately, how we will live within it."""

In [15]:
workflow.invoke({"essay":essay})

{'essay': 'in an age obsessed with grand innovations—artificial intelligence, space exploration, and genetic engineering—we rarely pause to examine the quiet, invisible structures that shape our everyday existence. From the layout of a grocery store designed to maximize impulse buying to the algorithmically curated social media feed that subtly dictates our mood, modern life is governed by what architect Christopher Alexander called a “pattern language.” These unseen architectures, while often benign, warrant closer scrutiny because they influence our choices, constrain our autonomy, and ultimately redefine what it means to act freely.\n\nFirst, consider the physical spaces we inhabit without thought. The suburban cul-de-sac, celebrated for its safety and community feel, simultaneously isolates residents by discouraging through-traffic and removing the chance encounters that define urban neighborhoods. Similarly, the modern open-plan office, intended to foster collaboration, often gene

Essay Input → [Depth Node || Clarity Node || Language Node] → Final Evaluation → Score